<a href="https://colab.research.google.com/github/Aje-dotcom/DeepTech-/blob/master/Ekene_Ajemba__Assignment1_week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Kindly find attached tables (find tables here) to answer the following
questions:
1. Write a query to list all customers and their transaction details.
Ensure customers without transactions and transactions without
matching customers are included.
2. Identify the customer(s) who have the highest total transaction
amount using a subquery.
3. Write a query to combine the list of customers from Lagos and Edo
using UNION, excluding duplicates.

In [7]:
import pandas as pd
import duckdb

file_path = '/content/Advanced SQL Queries Assignment 1.xlsx'

# The file has headers on the second row (index 1)
df_raw = pd.read_excel(file_path, sheet_name='Sheet1', header=1)

# Separate the tables and clean headers
# Customers table (top section), skipping the extra header row at index 0
Customers = df_raw.iloc[1:10, 0:4].dropna(how='all').copy()
Customers.columns = ['Customer_ID', 'Name', 'State', 'Income']
Customers['Customer_ID'] = pd.to_numeric(Customers['Customer_ID'])

# Transactions table (bottom section)
Transactions = pd.read_excel(file_path, sheet_name='Sheet1', header=14)
Transactions.columns = ['Transaction_ID', 'Customer_ID', 'Amount', 'Transaction_Type', 'Date']
Transactions = Transactions.dropna(subset=['Transaction_ID'])
Transactions['Customer_ID'] = pd.to_numeric(Transactions['Customer_ID'])
Transactions['Amount'] = pd.to_numeric(Transactions['Amount'])

print("Cleaned Customers Table:")
display(Customers.head())
print("\nCleaned Transactions Table:")
display(Transactions.head())

Cleaned Customers Table:


,Customer_ID,Name,State,Income
1,3021,Kolawale Saidu,Lagos,85000
2,3028,Ade Abu,Edo,120000
3,3067,Imabong Udo,Akwa Ibom,65000
4,3078,Diana Ross,Cross River,95000
5,3097,Adullahi Usman,Yobe,70000



Cleaned Transactions Table:


,Transaction_ID,Customer_ID,Amount,Transaction_Type,Date
0,T001,3021,8000,Credit,2024-12-01
1,T002,3028,1000,Debit,2024-12-02
2,T003,3078,4000,Credit,2024-12-03
3,T004,3067,1500,Credit,2024-12-03
4,T005,3021,15000,Debit,2024-12-04


In [11]:
# Execute the SQL query using DuckDB
query = """
SELECT
    c.Customer_ID,
    c.Name,
    c.State,
    t.Transaction_ID,
    t.Amount,
    t.Transaction_Type,
    t.Date
FROM Customers c
FULL OUTER JOIN Transactions t
    ON c.Customer_ID = t.Customer_ID;
"""

result = duckdb.query(query).to_df()
display(result)

,Customer_ID,Name,State,Transaction_ID,Amount,Transaction_Type,Date
0,3021,Kolawale Saidu,Lagos,T001,8000,Credit,2024-12-01
1,3028,Ade Abu,Edo,T002,1000,Debit,2024-12-02
2,3078,Diana Ross,Cross River,T003,4000,Credit,2024-12-03
3,3067,Imabong Udo,Akwa Ibom,T004,1500,Credit,2024-12-03
4,3021,Kolawale Saidu,Lagos,T005,15000,Debit,2024-12-04
5,3097,Adullahi Usman,Yobe,T006,30000,Debit,2024-12-05
6,3028,Ade Abu,Edo,T007,90000,Credit,2024-12-05
7,3056,Chidinma Ikena,Abia,T008,7600,Debit,2024-12-06
8,3043,Jefferson Chris,Taraba,T009,5800,Credit,2024-12-06


In [12]:
# 1. Write a query to list all customers and their transaction details.
query_1 = """
SELECT
    c.Customer_ID,
    c.Name,
    c.State,
    t.Transaction_ID,
    t.Amount,
    t.Transaction_Type,
    t.Date
FROM Customers c
FULL OUTER JOIN Transactions t
    ON c.Customer_ID = t.Customer_ID;
"""

result_1 = duckdb.query(query_1).to_df()
display(result_1)

,Customer_ID,Name,State,Transaction_ID,Amount,Transaction_Type,Date
0,3021,Kolawale Saidu,Lagos,T001,8000,Credit,2024-12-01
1,3028,Ade Abu,Edo,T002,1000,Debit,2024-12-02
2,3078,Diana Ross,Cross River,T003,4000,Credit,2024-12-03
3,3067,Imabong Udo,Akwa Ibom,T004,1500,Credit,2024-12-03
4,3021,Kolawale Saidu,Lagos,T005,15000,Debit,2024-12-04
5,3097,Adullahi Usman,Yobe,T006,30000,Debit,2024-12-05
6,3028,Ade Abu,Edo,T007,90000,Credit,2024-12-05
7,3056,Chidinma Ikena,Abia,T008,7600,Debit,2024-12-06
8,3043,Jefferson Chris,Taraba,T009,5800,Credit,2024-12-06


<b>1. List all customers and their transaction details (Full Outer Join)<br></b>
This query ensures that customers without transactions and transactions without matching customers are included in the results. <I>(Note: In databases like SQLite that do not support FULL OUTER JOIN directly, a UNION of a LEFT JOIN and a RIGHT JOIN was used to achieve the same result.)</I>

In [13]:
# 2. Identify the customer(s) who have the highest total transaction amount using a subquery.
query_2 = """
SELECT Name, Total_Amount
FROM (
    SELECT c.Name, SUM(t.Amount) as Total_Amount
    FROM Customers c
    JOIN Transactions t ON c.Customer_ID = t.Customer_ID
    GROUP BY c.Name
)
WHERE Total_Amount = (
    SELECT MAX(Total_Amount)
    FROM (
        SELECT SUM(Amount) as Total_Amount
        FROM Transactions
        GROUP BY Customer_ID
    )
);
"""

result_2 = duckdb.query(query_2).to_df()
display(result_2)

,Name,Total_Amount
0,Ade Abu,91000.0


<b>2. Customer(s) with the highest total transaction amount.</b><br>
This query uses a subquery to find the maximum sum of transaction amounts and then identifies the corresponding customer

In [14]:
# 3. Combine the list of customers from Lagos and Edo using UNION, excluding duplicates.
query_3 = """
SELECT * FROM Customers WHERE State = 'Lagos'
UNION
SELECT * FROM Customers WHERE State = 'Edo';
"""

result_3 = duckdb.query(query_3).to_df()
display(result_3)

,Customer_ID,Name,State,Income
0,3028,Ade Abu,Edo,120000
1,3021,Kolawale Saidu,Lagos,85000


<b>3. Combine customers from Lagos and Edo (UNION)</b><br>
This query combines the rows from both states while automatically excluding duplicates (as per the UNION operator behavior